# Chapter 01: API Elements of scikit-learn & Exploratory Data Analysis

# Identitas Praktikan
* **Nama**: Aryo Zaki Abdillah
* **NIM**: 101032300060
* **Kelas**: TK47-02
* **Program Studi**: S1 Teknik Komputer - Telkom University

---


## 1.1 Filosofi Desain dan Arsitektur API scikit-learn
scikit-learn dirancang dengan antarmuka yang sangat terstruktur dan konsisten:
- **Estimator**: Objek yang mempelajari parameter data melalui metode `.fit(X, y)`.
- **Transformer**: Objek yang menguba bentuk data melalui `.transform(X)` atau `.fit_transform(X)`.
- **Predictor**: Objek yang memprediksi nilai target melalui `.predict(X)` atau `.predict_proba(X)`.
- **Inspection**: Parameter terpelajari selalu diakhiri dengan garis bawah `_` (misalnya `coef_`, `intercept_`, `mean_`).

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

print("Import modul scikit-learn dan pendukung berhasil!")

## 1.2 Analisis Statistik Deskriptif & Deteksi Outlier (EDA)
Membandingkan ukuran pemusatan yang sensitif terhadap outlier (Mean) dengan yang kebal/robust (Median & Trimmed Mean).

In [ ]:
# Membuat dataset sintetis dengan nilai pencilan (outlier)
np.random.seed(42)
data_normal = np.random.normal(loc=100, scale=15, size=150)
data_outliers = np.array([450, -150, 500])
data_raw = np.concatenate([data_normal, data_outliers])

mean_val = np.mean(data_raw)
median_val = np.median(data_raw)
std_val = np.std(data_raw)
iqr_val = np.percentile(data_raw, 75) - np.percentile(data_raw, 25)

print(f"Mean (Sensitif Outlier) : {mean_val:.2f}")
print(f"Median (Robust)         : {median_val:.2f}")
print(f"Standar Deviasi (Std)   : {std_val:.2f}")
print(f"Interquartile Range(IQR): {iqr_val:.2f}")

## 1.3 Visualisasi Distribusi Data
Visualisasi boxplot dan histogram untuk melihat pencilan secara intuitif.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.boxplot(x=data_raw, ax=axes[0], color='lightskyblue')
axes[0].set_title("Boxplot Deteksi Outlier")

sns.histplot(data_raw, kde=True, ax=axes[1], color='salmon')
axes[1].set_title("Distribusi Data (Histogram)")

plt.tight_layout()
plt.show()

## 1.4 Pembuatan Custom Transformer (`OutlierClipper`)
Mewarisi `BaseEstimator` dan `TransformerMixin` untuk memangkas outlier secara otomatis di dalam pipeline.

In [ ]:
class OutlierClipper(BaseEstimator, TransformerMixin):
    def __init__(self, lower_quantile=0.05, upper_quantile=0.95):
        self.lower_quantile = lower_quantile
        self.upper_quantile = upper_quantile
        
    def fit(self, X, y=None):
        X_arr = np.asarray(X)
        self.lower_bound_ = np.quantile(X_arr, self.lower_quantile, axis=0)
        self.upper_bound_ = np.quantile(X_arr, self.upper_quantile, axis=0)
        return self
        
    def transform(self, X):
        X_arr = np.asarray(X)
        return np.clip(X_arr, self.lower_bound_, self.upper_bound_)

# Pengujian Transformer Kustom
clipper = OutlierClipper(lower_quantile=0.05, upper_quantile=0.95)
data_clipped = clipper.fit_transform(data_raw.reshape(-1, 1))
print("Lower Bound (5%) :", clipper.lower_bound_)
print("Upper Bound (95%):", clipper.upper_bound_)

## 1.5 Integrasi Pipeline End-to-End
Menggabungkan custom transformer, standard scaler, dan estimator regresi linier dalam 1 objek pipeline.

In [ ]:
X = np.random.randn(120, 3) * 10
y = 3.0 * X[:, 0] - 2.0 * X[:, 1] + 1.5 * X[:, 2] + np.random.randn(120)

pipe = Pipeline([
    ('clipper', OutlierClipper(lower_quantile=0.02, upper_quantile=0.98)),
    ('scaler', StandardScaler()),
    ('regressor', LinearRegression())
])

pipe.fit(X, y)
print("Koefisien Model (Beta) :", pipe.named_steps['regressor'].coef_)
print("Intercept              :", pipe.named_steps['regressor'].intercept_)
print("Skor R^2 Pipeline      :", pipe.score(X, y))